# Rebel on the Canal — Stata 复现 Notebook

**Cao & Chen (2022, AER)**: Disrupted Trade Access and Social Conflict in China, 1650–1911

---

## 使用说明

1. **需要 Stata 17+** 及 `stata_kernel`（`pip install stata_kernel && python -m stata_kernel.install`）
2. 先运行「环境检查」cell 安装外部命令（仅需一次）
3. 按顺序运行 PART A 的三个 cell（数据准备）
4. 之后每个图表 cell 可独立运行，顺序不限

## 数据获取

```bash
git clone https://github.com/Zhihan-iris/literature-reading.git
cd literature-reading/canal_data
```

或直接下载 `canal_rebellion.dta` 放入 `canal_data/` 文件夹。


In [1]:
/*===========================================================================
 * 0. 环境检查 & 外部命令安装（仅需运行一次）
 *===========================================================================*/

ssc install reghdfe
ssc install ftools
ssc install estout
ssc install ols_spatial_HAC
ssc install synth_runner
ssc install cic
ssc install matmap

display "所有外部命令安装完成。"
graph set window fontface "Cambria"
set scheme s2color


checking reghdfe consistency and verifying not already installed...


In [ ]:
/*===========================================================================
 * 1. 数据加载 & 被解释变量构建（PART A1-A2）
 *===========================================================================*/

* --- 请将路径改为你本机的 canal_data 文件夹位置 ---
cd "你的路径/canal_data"

* --- 加载数据 ---
use "canal_rebellion.dta", clear

* --- 扩大矩阵容量，高维固定效应回归需要 ---
set matsize 11000

* --- 声明面板结构：个体 = OBJECTID（县），时间 = year（年）---
xtset OBJECTID year

display "数据加载完成：" _N " 条观测, " _k " 个变量"

/*---------------------------------------------------------------------------
 * 被解释变量：反叛次数的标准化
 * 基准设定：asinh(onset_all / (cntypop1600 / 1000000))
 *---------------------------------------------------------------------------*/

*** 按土地面积标准化
gen lonset_km2   = ln(1 + onset_all / (AREA / 10000))
gen ashonset_km2 = asinh(onset_all / (AREA / 10000))

*** 按逐年插值人口标准化（六截面线性插值）
gen popden = popden1600 if year <= 1600
replace popden = popden1600 + (year - 1600) * ((popden1776 - popden1600) / (1776 - 1600)) ///
    if year > 1600 & year <= 1776
replace popden = popden1776 + (year - 1776) * ((popden1820 - popden1776) / (1820 - 1776)) ///
    if year > 1776 & year <= 1820
replace popden = popden1820 + (year - 1820) * ((popden1851 - popden1820) / (1851 - 1820)) ///
    if year > 1820 & year <= 1851
replace popden = popden1851 + (year - 1851) * ((popden1880 - popden1851) / (1880 - 1851)) ///
    if year > 1851 & year <= 1880
replace popden = popden1880 + (year - 1880) * ((popden1910 - popden1880) / (1910 - 1880)) ///
    if year > 1880 & year <= 1910
replace popden = popden1910 if year > 1910

gen pop     = popden     * AREA / 1000000
gen pop1600 = popden1600 * AREA / 1000000
gen pop1820 = popden1820 * AREA / 1000000

gen lonset_pop      = ln(1 + onset_all / pop)
gen lonset_pop1600  = ln(1 + onset_all / pop1600)
gen lonset_pop1820  = ln(1 + onset_all / pop1820)
gen ashonset_pop      = asinh(onset_all / pop)
gen ashonset_pop1600  = asinh(onset_all / pop1600)
gen ashonset_pop1820  = asinh(onset_all / pop1820)

*** 按插值县级人口（cntypop）标准化
gen cntypop = cntypop1600 if year <= 1600
replace cntypop = cntypop1600 + (year - 1600) * ((cntypop1776 - cntypop1600) / (1776 - 1600)) ///
    if year > 1600 & year <= 1776
replace cntypop = cntypop1776 + (year - 1776) * ((cntypop1820 - cntypop1776) / (1820 - 1776)) ///
    if year > 1776 & year <= 1820
replace cntypop = cntypop1820 + (year - 1820) * ((cntypop1851 - cntypop1820) / (1851 - 1820)) ///
    if year > 1820 & year <= 1851
replace cntypop = cntypop1851 + (year - 1851) * ((cntypop1880 - cntypop1851) / (1880 - 1851)) ///
    if year > 1851 & year <= 1880
replace cntypop = cntypop1880 + (year - 1880) * ((cntypop1910 - cntypop1880) / (1910 - 1880)) ///
    if year > 1880 & year <= 1910
replace cntypop = cntypop1910 if year > 1910

gen ashonset_cntypop      = asinh(onset_all / (cntypop / 1000000))
gen ashonset_cntypop1600  = asinh(onset_all / (cntypop1600 / 1000000))
gen ashonset_cntypop1820  = asinh(onset_all / (cntypop1820 / 1000000))

gen popdencnty1600    = cntypop1600 / AREA
gen lpopdencnty1600   = ln(popdencnty1600)

display "被解释变量构建完成。"
summarize ashonset_cntypop1600

In [ ]:
/*===========================================================================
 * 2. 控制变量 & 全局宏（PART A3-A5）
 *===========================================================================*/

sort OBJECTID year

*** 事前反叛趋势
by OBJECTID: egen prerebels = total((onset_all / (cntypop / 1000000)) * (reform == 0))
gen ashprerebels = asinh(prerebels)

*** 地理 x reform
gen rug_after    = ruggedness    * reform
gen huang_after  = alonghuang    * reform
gen yangtze_after = alongyangtze * reform

*** 气候 x reform
egen reconmean   = mean(recon)
egen reconsd     = sd(recon)
gen disaster     = (abs(recon - reconmean) > reconsd)
gen drought      = (climate == 1)
gen flooding     = (climate == 5)

*** 距离 x reform
gen distyellow_after = distance_huang * reform
gen distcoast_after  = distance_coast * 100 * reform

*** 基础地理 x reform
gen larea_after     = ln(AREA)        * reform
gen lpop1600_after  = ln(cntypop1600) * reform

*** 气候冲击 x reform
gen recon_after    = recon    * reform
gen drought_after  = drought  * reform
gen flooding_after = flooding * reform
gen disaster_after = disaster * reform

*** 农业 x reform
gen maize_after       = maize              * reform
gen sweetpotato_after = sweetpotato        * reform
gen wheat_after       = suitable_wheat_good * reform
gen rice_after        = suitable_rice_good  * reform
gen lwheat_after      = ln(si_wheat)       * reform
gen lrice_after       = ln(si_rice)        * reform

*** 人口密度 x reform
gen popdencnty1600_after  = popdencnty1600  * reform
gen lpopdencnty1600_after = lpopdencnty1600 * reform
gen taiping_after = Taiping * reform

*** 变量标签
label variable drought          "Drought"
label variable drought_after    "Drought $	imes$ Post"
label variable flooding         "Flooding"
label variable flooding_after   "Flooding $	imes$ Post"
label variable disaster         "Temperature Anomaly"
label variable disaster_after   "Temperature Anomaly $	imes$ Post"
label variable rug_after        "Ruggedness $	imes$ Post"
label variable interaction1     "Along Canal $	imes$ Post"

*** 全局宏
macro drop _all
est clear
global Y ashonset_cntypop1600
global X interaction1
gen area_after = AREA * reform
global ctrls larea_after rug_after disaster disaster_after flooding drought ///
       flooding_after drought_after lpopdencnty1600_after maize maize_after ///
       sweetpotato sweetpotato_after wheat_after rice_after
global stars nostar

display "控制变量与全局宏构建完成。"
display "Y = $Y"
display "X = $X" 

---
## Figure 2 · 第一阶段：漕运量的断裂

分段线性拟合 — 1826年前后漕运量的断崖式下降。


In [ ]:
/*=== Figure 2: 漕运量断裂 ===*/
preserve

duplicates drop year, force
keep if year > 1755 & year < 1860

#d ;
twoway
    (lfit lamount year if year <= 1825, lpattern(dash) lcolor("0 0 0"))
    (lfit lamount year if year >= 1826, lpattern(dash) lcolor("0 0 0"))
    (scatter lamount year, color("190 190 190") msize(*0.75))
    ,
    ytitle("Shipping volume (log million piculs)", size(*0.9))
    xtitle("")
    yline(0.8(0.1)1.8, lstyle(grid) lwidth(thin) lcolor("235 235 235"))
    xline(1760(10)1860, lstyle(grid) lwidth(thin) lcolor("235 235 235"))
    xline(1825.3, lpattern(dash) lcolor("128 0 0"))
    ylabel(0.8(0.2)1.8, angle(0) format(%5.1f) labsize(*0.85))
    xlabel(, labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(off)
    ;
#d cr
graph export "fig_canal_usage.png", replace

restore

## Figure 4 · 事件研究：运河废弃与反叛的动态效应

10年分箱事件研究，-60期为基准组。改革前系数围绕零，改革后系统性跃升。


In [ ]:
/*=== Figure 4: 事件研究 ===*/
preserve

gen aperiod = floor((year - 1826) / 10) * 10
replace aperiod = -60 if aperiod < -60
tab aperiod, gen(aperiod)
keep if aperiod < 70

reghdfe $Y c.alongcanal#(c.aperiod2-aperiod15) $ctrls, ///
    absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)

matrix coef = e(b)
matrix cov  = e(V)
gen coef = .
gen se   = .
forvalues i = 1(1)12 {
    replace coef = coef[1, `i'] if _n == `i'
    replace se   = sqrt(cov[`i', `i']) if _n == `i'
}
gen lb = coef - invttail(e(df_r), 0.025) * se
gen ub = coef + invttail(e(df_r), 0.025) * se
keep coef se lb ub
drop if coef == .
gen year = _n

#d ;
twoway
    (rarea ub lb year, color("193 205 205%80"))
    (scatter coef year, color(gs0) msize(*0.75))
    (line coef year, lpattern(solid) lcolor("4 4 4"))
    ,
    ytitle("Coefficients", size(*0.9))
    xtitle("Number of years since the 1826 reform", size(*0.9) margin(medsmall))
    yline(-0.05(0.025)0.2, lstyle(grid) lwidth(thin) lcolor("235 235 235"))
    xline(1(0.5)12, lstyle(grid) lwidth(thin) lcolor("235 235 235"))
    yline(0, lpattern(dash) lcolor("128 0 0"))
    xline(5.5, lpattern(dash) lcolor("128 0 0"))
    ylabel(-0.05(0.05)0.2, angle(0) format(%5.2f) labsize(*0.85))
    xlabel(1 "-50" 2 "-40" 3 "-30" 4 "-20" 5 "-10"
           6 "10" 7 "20" 8 "30" 9 "40" 10 "50" 11 "60" 12 "70", labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(off)
    ;
#d cr
graph export "fig_event_study.png", replace

restore

## Table 2 · 事前趋势检验

样本限制在1776-1825年，检验 `alongcanal x year` 是否显著。不显著 -> 平行趋势成立。

四列递进：县FE+年FE -> +事前反叛趋势x年FE -> +省x年FE -> +府x线性趋势。


In [ ]:
/*=== Table 2: 事前趋势检验 ===*/
preserve

keep if year >= 1776 & year <= 1825
gen pretrend = alongcanal * year
label variable pretrend "$ Along Canal \times Year $ "
global X pretrend

forvalues c = 1/4 {
    if `c' == 1 local fes i.OBJECTID i.year
    if `c' == 2 local fes i.OBJECTID i.year c.ashprerebels#i.year
    if `c' == 3 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year
    if `c' == 4 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year i.prefid#c.year
    reghdfe $Y $X, absorb(`fes') cluster(OBJECTID)
    eststo est`c'
    qui tab OBJECTID if e(sample)
    scalar groups = r(r)
    qui su $Y if e(sample)
    scalar ymean = r(mean)
    estadd scalar depavg = ymean : est`c'
    estadd scalar N_g    = groups : est`c'
    
    preserve
        hdfe $Y $X, clear absorb(`fes') tol(0.001) keepvars(OBJECTID year Y_COORD X_COORD)
        ols_spatial_HAC $Y $X, lat(Y_COORD) lon(X_COORD) time(year) ///
            panel(OBJECTID) distcutoff(500) lagcutoff(50) disp star
        matrix V_spat = vecdiag(e(V))
        matmap V_spat SE_spat, m(sqrt(@))
        estadd matrix sesp = SE_spat : est`c'
    restore
}

estfe est1 est2 est3 est4
esttab est1 est2 est3 est4, keep(pretrend) $stars se ///
    stats(depavg N_g N, labels("Mean of depvar" "Number of groups" "Observations"))

restore

## Table 3 · 基准回归：运河废弃与反叛

**论文最核心的表格。** 五列递进：
1. 县FE + 年FE
2. + 事前反叛趋势 x 年FE
3. + 省 x 年FE
4. + 府 x 线性趋势
5. + 全部控制变量

每列接 Conley 空间标准误（500km, 262年）。约需 10-15 分钟。


In [ ]:
/*=== Table 3: 基准回归 ===*/
preserve

forvalues c = 1/5 {
    if `c' == 1 local fes i.OBJECTID i.year
    if `c' == 2 local fes i.OBJECTID i.year c.ashprerebels#i.year
    if `c' == 3 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year
    if `c' == 4 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year i.prefid#c.year
    if `c' == 5 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year i.prefid#c.year
    local ctrl
    if `c' == 5 local ctrl $ctrls
    
    reghdfe $Y $X `ctrl', absorb(`fes') cluster(OBJECTID)
    eststo est`c'
    qui tab OBJECTID if e(sample)
    scalar groups = r(r)
    qui su $Y if e(sample)
    scalar ymean = r(mean)
    estadd scalar depavg = ymean : est`c'
    estadd scalar N_g    = groups : est`c'
    
    preserve
        hdfe $Y $X `ctrl', clear absorb(`fes') tol(0.001) keepvars(OBJECTID year Y_COORD X_COORD)
        ols_spatial_HAC $Y $X, lat(Y_COORD) lon(X_COORD) time(year) ///
            panel(OBJECTID) distcutoff(500) lagcutoff(262) disp star
        matrix V_spat = vecdiag(e(V))
        matmap V_spat SE_spat, m(sqrt(@))
        estadd matrix sesp = SE_spat : est`c'
    restore
}

estfe est1 est2 est3 est4 est5
esttab est1 est2 est3 est4 est5, keep(interaction1) $stars se ///
    stats(depavg N_g N, labels("Mean of depvar" "Number of groups" "Observations"))

restore

## Table 4 · 处理强度：剂量-反应关系

三个连续处理变量：运河密度(+)、运河城镇占比(+)、距运河距离(-)。


In [ ]:
/*=== Table 4: 剂量-反应 ===*/
preserve

gen ash_den  = asinh(canal_den)
gen ash_dist = asinh(ctadmin_canal)
gen ash_den_after  = ash_den  * reform
gen ash_dist_after = ash_dist * reform
gen canaltown_after = (town1820_r10canal * alongcanal) / town1820 * reform

global X1 ash_den_after
global X2 canaltown_after
global X3 ash_dist_after

foreach est in int1 int2 int3 {
    if "`est'" == "int1" local Xvar $X1
    if "`est'" == "int2" local Xvar $X2
    if "`est'" == "int3" local Xvar $X3
    reghdfe $Y `Xvar', absorb(i.OBJECTID i.year c.ashprerebels#i.year ///
        i.provid#i.year i.prefid#c.year) cluster(OBJECTID)
    estimates store `est'
    qui tab OBJECTID if e(sample)
    scalar groups = r(r)
    su $Y if e(sample)
    scalar ymean = r(mean)
    estadd scalar depavg = ymean : `est'
    estadd scalar N_g    = groups : `est'
    
    preserve
        hdfe $Y `Xvar', clear absorb(i.OBJECTID i.year c.ashprerebels#i.year ///
            i.provid#i.year i.prefid#c.year) tol(0.001) keepvars(OBJECTID year Y_COORD X_COORD)
        ols_spatial_HAC $Y `Xvar', lat(Y_COORD) lon(X_COORD) time(year) ///
            panel(OBJECTID) distcutoff(500) lagcutoff(262) disp star
        matrix V_spat = vecdiag(e(V))
        matmap V_spat SE_spat, m(sqrt(@))
        estadd matrix sesp = SE_spat : `est'
    restore
}

estfe int1 int2 int3
esttab int1 int2 int3, $stars se stats(depavg N_g N)

restore

## Table 5 · 南北异质性

以旧黄河与运河交点纬度分南北。三重交互 `alongcanal x reform x north`。预期：北方 > 南方。


In [ ]:
/*=== Table 5: 南北异质性 ===*/
preserve

gen intlat = Y_COORD if along_oldhuang == 1 & alongcanal == 1
egen intersectlat = min(intlat)
gen north = (Y_COORD > intersectlat)
gen northpost = north * reform
gen triple = alongcanal * reform * north
global X triple interaction1 northpost

forvalues c = 1/4 {
    if `c' == 1 local fes i.OBJECTID i.year
    if `c' == 2 local fes i.OBJECTID i.year c.ashprerebels#i.year
    if `c' == 3 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year
    if `c' == 4 local fes i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year i.prefid#c.year
    reghdfe $Y $X, absorb(`fes') cluster(OBJECTID)
    eststo est`c'
    qui tab OBJECTID if e(sample)
    scalar groups = r(r)
    qui su $Y if e(sample)
    scalar ymean = r(mean)
    estadd scalar depavg = ymean : est`c'
    estadd scalar N_g    = groups : est`c'
    
    preserve
        hdfe $Y $X, clear absorb(`fes') tol(0.001) keepvars(OBJECTID year Y_COORD X_COORD)
        ols_spatial_HAC $Y $X, lat(Y_COORD) lon(X_COORD) time(year) ///
            panel(OBJECTID) distcutoff(500) lagcutoff(262) disp star
        matrix V_spat = vecdiag(e(V))
        matmap V_spat SE_spat, m(sqrt(@))
        estadd matrix sesp = SE_spat : est`c'
    restore
}

estfe est1 est2 est3 est4
esttab est1 est2 est3 est4, keep(triple) $stars se stats(depavg N_g N)

restore

## Table 6 · 安慰剂检验：替代交通路线

长江、旧黄河、海岸线、驿道——四条伪处理路线。预期全部不显著。


In [ ]:
/*=== Table 6: 安慰剂检验 ===*/
preserve

gen oldhuang_after = along_oldhuang * reform
gen coast_after    = alongcoast    * reform
gen courier_after  = alongcourier  * reform
egen plcb = rowmax(alongyangtze along_oldhuang alongcoast alongcourier)
gen plcb_after = plcb * reform

global Xp1    yangtze_after
global Xp2    oldhuang_after
global Xp3    coast_after
global Xp4    courier_after
global Xp_any plcb_after

foreach x in Xp1 Xp2 Xp3 Xp4 Xp_any {
    local col = subinstr("`x'", "Xp", "", .)
    if "`col'" == "_any" local col = 5
    reghdfe $Y ${`x'}, absorb(i.OBJECTID i.year c.ashprerebels#i.year ///
        i.provid#i.year i.prefid#c.year) cluster(OBJECTID)
    eststo est`col'
    qui tab OBJECTID if e(sample)
    scalar groups = r(r)
    qui su $Y if e(sample)
    scalar ymean = r(mean)
    estadd scalar depavg = ymean : est`col'
    estadd scalar N_g    = groups : est`col'
    
    preserve
        hdfe $Y ${`x'}, clear absorb(i.OBJECTID i.year c.ashprerebels#i.year ///
            i.provid#i.year i.prefid#c.year) tol(0.001) keepvars(OBJECTID year Y_COORD X_COORD)
        ols_spatial_HAC $Y ${`x'}, lat(Y_COORD) lon(X_COORD) time(year) ///
            panel(OBJECTID) distcutoff(500) lagcutoff(262) disp star
        matrix V_spat = vecdiag(e(V))
        matmap V_spat SE_spat, m(sqrt(@))
        estadd matrix sesp = SE_spat : est`col'
    restore
}

estfe est1 est2 est3 est4 est5
esttab est1 est2 est3 est4 est5, $stars se stats(depavg N_g N)

restore

## Table 7 · 排除战争干扰

**Panel A**: 剔除鸦片战争战场县 & 太平天国核心区。

**Panel B**: 三重交互 `alongcanal x reform x warzone`。


In [ ]:
/*=== Table 7: 排除战争干扰 ===*/
preserve

gen taipingregion = (Taiping >= 2) if Taiping < .
gen triple1  = alongcanal * reform * opiumbattle
gen triple1a = opiumbattle * reform
gen triple2  = alongcanal * reform * taipingregion
gen triple2a = taipingregion * reform

* --- Panel A: 剔除鸦片战争战场县 ---
preserve
    keep if opiumbattle == 0
    reghdfe $Y $X, absorb(OBJECTID year c.ashprerebels#i.year i.prefid#c.year i.provid#i.year) cluster(OBJECTID)
    eststo omtopium1
restore

* --- Panel A: 剔除太平天国核心区 ---
preserve
    keep if Taiping < 2
    reghdfe $Y $X, absorb(OBJECTID year c.ashprerebels#i.year i.prefid#c.year i.provid#i.year) cluster(OBJECTID)
    eststo omttaiping1
restore

* --- Panel B: 三重交互 ---
reghdfe $Y interaction1 triple1a triple1, absorb(i.OBJECTID i.year ///
    c.ashprerebels#i.year i.prefid#c.year i.provid#i.year) cluster(OBJECTID)
est store opium1

reghdfe $Y interaction1 triple2a triple2, absorb(i.OBJECTID i.year ///
    c.ashprerebels#i.year i.prefid#c.year i.provid#i.year) cluster(OBJECTID)
est store taiping1

esttab omtopium1 omttaiping1 opium1 taiping1, keep(interaction1) $stars se

restore

## Figure A1 · 反叛次数的时间分布

1650-1911年每年反叛总次数。


In [ ]:
/*=== Figure A1: 反叛时间分布 ===*/
preserve

collapse (sum) onset_all, by(year)
label variable onset_all "Count of Rebellions"
#d ;
twoway
    (line onset_all year, lwidth(*1.05) lcolor(gray*2.6))
    (scatter onset_all year, color(gs0) msize(*0.15))
    ,
    ytitle("Count of Rebellions", size(*0.9))
    xtitle("")
    xline(1826, lpattern(dash) lcolor("128 0 0"))
    ylabel(0(20)60, angle(0) format(%12.0f) labsize(*0.85))
    xlabel(1650(50)1900, labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(off)
    ;
#d cr
graph export "fig_rebellion_timeline.png", replace

restore

## Figure A3 · 灵活处理强度

运河密度和城镇占比各分5组，binned估计。


In [ ]:
/*=== Figure A3: 灵活处理强度 ===*/
preserve

gen canaltown = (town1820_r10canal * alongcanal) / town1820

* --- Panel (a)：运河密度分组 ---
preserve
    egen intensity = cut(canal_den), at(0, 0.001, 2, 4, 6, 15)
    replace intensity = -1 if alongcanal == 0
    tab intensity, gen(inten)
    reghdfe $Y c.reform#(c.inten2-inten6), absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
    matrix coef = e(b)
    matrix cov  = e(V)
    gen coef = .
    gen se   = .
    forvalues i = 1(1)5 {
        replace coef = coef[1, `i'] if _n == `i'
        replace se   = sqrt(cov[`i', `i']) if _n == `i'
    }
    gen lb = coef - invttail(e(df_r), 0.025) * se
    gen ub = coef + invttail(e(df_r), 0.025) * se
    keep coef se lb ub
    drop if coef == .
    gen ph_intensity = _n
    #d ;
    twoway
        (line lb ph_intensity, lpattern(dash) lcolor("0 0 0"))
        (line ub ph_intensity, lpattern(dash) lcolor("0 0 0"))
        (scatter coef ph_intensity, color(gs0) msize(*0.75))
        (line coef ph_intensity, lpattern(solid) lcolor("4 4 4"))
        ,
        ytitle("Coefficients", size(*0.9))
        xtitle("Canal length per 100 square km", size(*0.9))
        yline(0, lpattern(dash) lcolor("128 0 0"))
        ylabel(-0.05(0.05)0.10, angle(0) format(%5.2f) labsize(*0.85))
        xlabel(1 "0" 2 "2" 3 "4" 4 "6" 5 "6+", labsize(*0.85))
        graphregion(fcolor(gs16) lcolor(gs16))
        plotregion(lcolor("white") lwidth(*0.9))
        legend(off)
        ;
    #d cr
    graph export "fig_flexible_density.png", replace
restore

* --- Panel (b)：运河城镇占比分组 ---
preserve
    egen intensity = cut(canaltown), at(0, .2, .4, .6, .8, 1.1)
    replace intensity = -1 if alongcanal == 0
    tab intensity, gen(inten)
    reghdfe $Y c.reform#(c.inten2-inten6), absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
    matrix coef = e(b)
    matrix cov  = e(V)
    gen coef = .
    gen se   = .
    forvalues i = 1(1)5 {
        replace coef = coef[1, `i'] if _n == `i'
        replace se   = sqrt(cov[`i', `i']) if _n == `i'
    }
    gen lb = coef - invttail(e(df_r), 0.025) * se
    gen ub = coef + invttail(e(df_r), 0.025) * se
    keep coef se lb ub
    drop if coef == .
    gen ph_intensity = _n
    #d ;
    twoway
        (line lb ph_intensity, lpattern(dash) lcolor("0 0 0"))
        (line ub ph_intensity, lpattern(dash) lcolor("0 0 0"))
        (scatter coef ph_intensity, color(gs0) msize(*0.75))
        (line coef ph_intensity, lpattern(solid) lcolor("4 4 4"))
        ,
        ytitle("Coefficients", size(*0.9))
        xtitle("Share of towns within 10km to the canal", size(*0.9))
        yline(0, lpattern(dash) lcolor("128 0 0"))
        ylabel(-0.05(0.05)0.15, angle(0) format(%5.2f) labsize(*0.85))
        xlabel(1 "0.2" 2 "0.4" 3 "0.6" 4 "0.8" 5 "1.0", labsize(*0.85))
        graphregion(fcolor(gs16) lcolor(gs16))
        plotregion(lcolor("white") lwidth(*0.9))
        legend(off)
        ;
    #d cr
    graph export "fig_flexible_townshare.png", replace
restore

restore

## Figure A4 · 灵活距离：效应随距离衰减

按距运河25km分箱（16组），预期系数随距离递减。


In [ ]:
/*=== Figure A4: 距离衰减 ===*/
preserve

gen band = ceil(ctadmin_canal / 25) * 25
replace band = 425 if band >= 425 & band != .
tab band, gen(dist)
reghdfe $Y c.reform#(c.dist1-dist16), absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
matrix coef = e(b)
matrix cov  = e(V)
gen coef = .
gen se   = .
forvalues i = 1(1)16 {
    replace coef = coef[1, `i'] if _n == `i'
    replace se   = sqrt(cov[`i', `i']) if _n == `i'
}
gen lb = coef - invttail(e(df_r), 0.025) * se
gen ub = coef + invttail(e(df_r), 0.025) * se
keep coef se lb ub
drop if coef == .
gen distance_canal = _n
#d ;
twoway
    (line lb distance_canal, lpattern(dash) lcolor("0 0 0"))
    (line ub distance_canal, lpattern(dash) lcolor("0 0 0"))
    (scatter coef distance_canal, color(gs0) msize(*0.75))
    (line coef distance_canal, lpattern(solid) lcolor("4 4 4"))
    ,
    ytitle("Coefficients", size(*0.9))
    xtitle("Distance to the canal (km)", size(*0.9))
    yline(0, lpattern(dash) lcolor("128 0 0"))
    ylabel(-0.05(0.05)0.15, angle(0) format(%5.2f) labsize(*0.85))
    xlabel(1 "25" 2 "50" 3 "75" 4 "100" 5 "125" 6 "150" 7 "175" 8 "200"
        9 "225" 10 "250" 11 "275" 12 "300" 13 "325" 14 "350" 15 "375" 16 "400", labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(off)
    ;
#d cr
graph export "fig_distance_decay.png", replace

restore

## Figure A6 · 合成控制法 (SCM)

**Panel (a)**: 运河县实际反叛路径 vs 合成控制。**Panel (b)**: 处理效应 + 排列检验p值。

控制池限制：非运河县且距运河至少150km。


In [ ]:
/*=== Figure A6: 合成控制法 ===*/
preserve

* --- 折叠为十年均值 ---
replace year = floor((year - 1826) / 10) * 10 + 1826
collapse (mean) onset_all cntypop1600 alongcanal distance_canal, by(OBJECTID year)
gen ashonset_cntypop1600 = asinh(onset_all / (cntypop1600 / 1000000))
gen y = ashonset_cntypop1600
keep if y < .
keep if year >= 1776
drop if distance_canal < 150 & alongcanal == 0
gen interaction1 = alongcanal * (year >= 1826)

* --- 运行合成控制法 ---
synth_runner y y(1776) y(1796) y(1806) y(1816), d(interaction1) gen_var
matrix P = e(pvals_std)

* --- 提取p值 ---
preserve
    clear
    svmat P, names(matcol)
    gen I = 1
    reshape long Pc, i(I) j(lead)
    drop I
    rename Pc p_vals
    tempfile temp
    save "`temp'.dta", replace
restore
merge m:1 lead using "`temp'.dta", nogenerate
save "synth10alt.dta", replace

* --- Panel (a)：实际 vs 合成路径 ---
use "synth10alt.dta", clear
replace year = year - 1826
keep if alongcanal == 1
collapse (mean) p_vals y y_synth, by(year)
gen effect = y - y_synth
keep if year < 70
#d ;
twoway
    (connected y year, lpattern(solid)  msymbol(C) msize(*0.75) color("4 4 4"))
    (connected y_synth year, lpattern(dash) msymbol(T) msize(*0.75) color("119 119 119"))
    ,
    ytitle("Coefficients", size(*0.9))
    xtitle("Number of years since the 1826 reform", size(*0.9))
    xline(-5, lpattern(dash) lcolor("128 0 0"))
    ylabel(0(0.2)0.8, angle(0) format(%5.1f) labsize(*0.85))
    xlabel(-50 "-50" -40 "-40" -30 "-30" -20 "-20" -10 "-10"
        0 "10" 10 "20" 20 "30" 30 "40" 40 "50" 50 "60" 60 "70", labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(label(1 "Canal counties (treated)") label(2 "Synthetic controls") size(*0.85))
    ;
#d cr
graph export "fig_scm_path.png", replace

* --- Panel (b)：处理效应 + p值双轴图 ---
#d ;
twoway
    (line effect year, yaxis(1) lpattern(solid) color("4 4 4"))
    (scatter p_vals year, yaxis(2) msymbol(O) msize(*0.9) color("4 4 4"))
    ,
    ytitle("Treatment effects", axis(1) size(*0.9))
    ytitle("p-values", axis(2) size(*0.9))
    xtitle("Number of years since the 1826 reform", size(*0.9))
    xline(-5, lpattern(dash) lcolor("128 0 0"))
    ylabel(-0.2(0.2)0.8, angle(0) format(%5.1f) labsize(*0.85) axis(1))
    ylabel(0(0.2)1, angle(0) format(%5.1f) labsize(*0.85) axis(2))
    xlabel(-50 "-50" -40 "-40" -30 "-30" -20 "-20" -10 "-10"
        0 "10" 10 "20" 20 "30" 30 "40" 40 "50" 50 "60" 60 "70", labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(label(1 "Treatment effects") label(2 "P-values") size(*0.85))
    ;
#d cr
graph export "fig_scm_effects.png", replace

restore

## Figure A7 · 距长江距离的调节效应

100km带宽滑动窗口。预期：离长江越近 -> 运河效应越弱。


In [ ]:
/*=== Figure A7: 长江异质性 ===*/
preserve

mat result = (., ., ., .)
capture program drop svresult
program define svresult
    args distance_yangtze
    qui {
        reghdfe $Y $X $ctrls ///
            if distance_yangtze >= `distance_yangtze' - 100 ///
            & distance_yangtze < `distance_yangtze', ///
            a(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year i.prefid#c.year) cl(OBJECTID)
        scalar b      = _b[inter]
        scalar se     = _se[inter]
        scalar uci_95 = _b[inter] + invttail(e(df_r), 0.025) * se
        scalar lci_95 = _b[inter] - invttail(e(df_r), 0.025) * se
        mat result = (result \ `distance_yangtze', b, lci_95, uci_95)
    }
end

_dots 0
forvalues i = 100(100)400 {
    svresult `i'
    _dots `i' 0
}

clear
svmat result
drop in 1
rename result1 dist
rename result2 coef
rename result3 lci_95
rename result4 rci_95
export delimited "distance_yangtze.txt", delimit(tab) replace
import delimited using "distance_yangtze.txt", clear
#d ;
twoway
    (rcap rci_95 lci_95 dist, lcolor(Gray) lpattern(dash) lwidth(thin) msize(*0.75))
    (scatter coef dist, color(gs0) msize(*0.75))
    ,
    ytitle("Coefficients", size(*0.9))
    xtitle("Distance to the Yangtze river", size(*0.9))
    yline(0, lpattern(dash) lcolor("128 0 0"))
    ylabel(0(0.1)0.3, angle(0) format(%5.1f) labsize(*0.85))
    xlabel(, labsize(*0.85))
    graphregion(fcolor(gs16) lcolor(gs16))
    plotregion(lcolor("white") lwidth(*0.9))
    legend(off)
    ;
#d cr
graph export "fig_yangtze_heterogeneity.png", replace

restore

## Table A1 · 平衡性检验

截面比较：运河县 vs 非运河县。


In [ ]:
/*=== Table A1: 平衡性检验 ===*/
preserve

preserve
    gen larea = ln(AREA)
    gen lpop  = ln(popdencnty1600)
    global balancevars larea ruggedness disaster flooding drought lpop maize sweetpotato suitable_wheat_good suitable_rice_good
    collapse (mean) alongcanal $balancevars, by(OBJECTID)
restore

recode alongcanal (1=0) (0=1), gen(sugroup)
keep if lpop < .
estpost summarize $balancevars if sugroup == 0
eststo des1
estpost summarize $balancevars if sugroup == 1
eststo des2
estpost ttest $balancevars, by(sugroup)
eststo des3

restore

## Table A4 · 机制一：国家控制力

驻军、府治所在地三重交互 + 攻防类型区分。


In [ ]:
/*=== Table A4: 国家控制力机制 ===*/
preserve

gen ashdensity = asinh(canal_den)
global canal ashdensity
gen ashattack_cntypop1600  = asinh(attack  / (cntypop1600 / 1000000))
gen ashretreat_cntypop1600 = asinh(runinto / (cntypop1600 / 1000000))
gen canal_after = ${canal} * reform

* --- 驻军三重交互 ---
gen triplesoldier = ${canal} * asinh(soldier / 100) * reform
gen soldier_after = asinh(soldier / 100) * reform
reghdfe $Y canal_after soldier_after triplesoldier, absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
eststo vul1

* --- 府治所在地三重交互 ---
gen triplecapital = ${canal} * pref_capital * reform
gen capital_after = pref_capital * reform
reghdfe $Y canal_after capital_after triplecapital, absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
eststo vul2

* --- 攻击型 vs 退守型 ---
reghdfe ashattack_cntypop1600 canal_after, absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
eststo vul3
reghdfe ashretreat_cntypop1600 canal_after, absorb(i.OBJECTID i.year c.ashprerebels#i.year i.provid#i.year) cluster(OBJECTID)
eststo vul4

estfe vul1 vul2 vul3 vul4
esttab vul1 vul2 vul3 vul4, keep(canal_after) $stars se

restore

## Table A5 · 机制二：贸易通道

城镇数量变化 + 驿道替代 + 气候冲击 x 粮价渠道。


In [ ]:
/*=== Table A5: 贸易通道机制 ===*/
preserve

gen ashdensity    = asinh(canal_den)
gen ashcourierden = asinh(courier_length / (AREA / 10000))
global canal   ashdensity
global courier ashcourierden

* --- 城镇数量（1820->1911 两期DID）---
preserve
    duplicates drop OBJECTID, force
    drop year
    rename town1820_r10canal  canalside1820
    rename town1911_r10canal  canalside1911
    rename town1820_r10courier courierside1820
    rename town1911_r10courier courierside1911
    reshape long town canalside courierside, i(OBJECTID) j(year)
    gen canal_after = ${canal} * (year == 1911)
    replace town = ln(town)
    reghdfe town canal_after, absorb(i.OBJECTID i.year) cluster(OBJECTID)
    eststo trade1
restore

* --- 驿道替代效应 ---
gen canal_after = ${canal} * reform
gen triplecourier = ${canal} * reform * ${courier}
gen courier_after = reform * ${courier}
reghdfe $Y canal_after courier_after triplecourier, absorb(i.OBJECTID i.year) cluster(OBJECTID)
eststo trade2

* --- 气候冲击 x 运河依赖 ---
gen tripledisaster = ${canal} * reform * disaster
gen disaster_canal = ${canal} * disaster
reghdfe $Y canal_after disaster disaster_canal disaster_after tripledisaster, absorb(i.OBJECTID i.year) cluster(OBJECTID)
eststo trade3

estfe trade1 trade2 trade3
esttab trade1 trade2 trade3, keep(canal_after) $stars se

restore

## Table A7 · 青帮与运河废弃

截面回归：运河沿线县的青帮高级成员数。


In [ ]:
/*=== Table A7: 青帮 ===*/
preserve

collapse (mean) green_senior alongcanal prefid provid, by(OBJECTID)
global canal alongcanal
xi: reg green_senior $canal i.prefid, robust
eststo pst1
su green_senior if e(sample)
scalar ymean = r(mean)
estadd scalar depavg = ymean : pst1

esttab pst1, keep(_I*) $stars se

restore

---
## 运行检查清单

- [ ] Cell 1: 外部命令安装（仅一次）
- [ ] Cell 2-3: 数据准备（跑完才能运行后续分析块）
- [ ] 后续 Cell: 按需逐一运行，每块独立

**预期输出**：Table 3 中 `interaction1` 系数约为 0.08-0.12，Conley SE 约 0.02-0.03，t值 > 3。

**异常处理**：
- `reghdfe` 报 singular -> 检查固定效应是否共线，尝试去掉 `i.prefid#c.year`
- `ols_spatial_HAC` 报错 -> 确认安装了最新版（`ssc install ols_spatial_HAC, replace`）
- `cic` 报错 -> 确认 Stata >= 15，尝试 `ssc install cic, replace`
- `synth_runner` 长时间运行 -> 正常现象，5-15分钟

完整 `.do` 文件版见 `canal_analysis.do`。
